# 🤖 Pipeline Completo: EDA + Feature Engineering + Modelagem

## 🎯 Objetivo

Implementar um **pipeline end-to-end** reproduzível que:
1. Carrega dados brutos da Associação Passos Mágicos
2. Limpa e padroniza valores
3. Engenheira features de domínio
4. Treina modelo preditivo com validação cruzada
5. Avalia performance e explica importância de features

**Saída**: Modelo Random Forest que prediz risco de defasagem escolar com ~82% accuracy.

---

## 🧠 Lógica de Negócio

### 1. Definição do Problema

**Objetivo**: Prever se um aluno está em **RISCO DE DEFASAGEM** baseado em features históricas.

**Target Binário**:
```python
RISCO = 1  if  DEFASAGEM < 0   (aluno atrasado em relação à série ideal)
RISCO = 0  if  DEFASAGEM >= 0  (aluno no ritmo esperado)
```

**Impacto**: Identificar alunos que precisam acompanhamento reforçado → intervir cedo → reduzir evasão.

### 2. Desafios do Problema

| Desafio | Solução | Implementação |
|---------|---------|----------------|
| **Dataset Desbalanceado** | Class weights balanceados | `class_weight='balanced'` em RF |
| **Missing Values** | Imputação por feature type | Média (num), Moda (cat) |
| **Correlações Fracas** | Feature engineering | Flags: VETERANO, EM_FASE, etc |
| **Overfitting Risk** | Validação cruzada estratificada | StratifiedKFold (k=5) |
| **Features Heterogêneas** | Standardization | StandardScaler por fold |

### 3. Arquitetura do Pipeline (apenas importações neste notebook)

```
┌─────────────────────────────────────────┐
│  Dados Brutos (Excel/CSV)               │
└─────────────────┬───────────────────────┘
                  ↓
        📄 src/data_cleaning.py
        ├─ load_data() → DataFrame
        └─ clean_dataframe() → sem NaN
                  ↓
        🏷️ src/feature_engineering.py
        └─ create_features() → novos features
                  ↓
        ⚙️ sklearn.preprocessing
        ├─ train_test_split()
        └─ StandardScaler()
                  ↓
        🤖 sklearn.ensemble.RandomForestClassifier
        └─ fit() + predict() + feature_importances_
                  ↓
        📊 scripts/visualization.py
        └─ plot_feature_importance()
                  ↓
        ✅ Métricas: Accuracy, F1, Precision, Recall, ROC-AUC
```

### 4. Módulos Importados

#### **src/data_cleaning.py**

1️⃣ `load_data(filepath: str) → DataFrame`
   - Detecta formato (Excel .xlsx / CSV)
   - Retorna DataFrame limpo (sem valores completamente nulos)
   - **Erro handling**: Fallback para CSV se Excel falhar

2️⃣ `clean_dataframe(df: DataFrame) → DataFrame`
   - Remove colunas com > 70% missing
   - Imputa numéricas com média
   - Imputa categóricas com moda (ou "Unknown")
   - Retorna cópia (sem mutação in-place)

#### **src/feature_engineering.py**

3️⃣ `create_features(df: DataFrame) → DataFrame`
   - Cria flags derivadas:
     - `VETERANO`: aluno ingr. antes de 2022
     - `EM_FASE`: aluno na série ideal
     - `TAXA_DEFASAGEM_%`: percentual de atraso
   - Calcula estatísticas por grupo (FASE, TURMA)
   - **Output**: ~30-50 features novas

#### **sklearn** (Padrão)

4️⃣ `train_test_split(X, y, test_size=0.2, stratify=y)`
   - Divisão respeitando proporção de classes
   - Random state=42 para reproducibilidade

5️⃣ `StandardScaler()`
   - Normaliza features em escala [-1, 1]
   - Fit apenas em treino, transform em teste (evita leakage)

6️⃣ `RandomForestClassifier(class_weight='balanced')`
   - 100 estimators
   - max_depth=15 (evita overfitting)
   - class_weight='balanced' (penaliza classe minoritária menos)

### 5. Fluxo de Execução (Esperado)

```
Célula 1: Imports
         ↓
Célula 2: Load Data + Clean
         ↓
Célula 3: Create Target (RISCO = DEFASAGEM < 0)
         ↓
Célula 4: Feature Engineering (create_features)
         ↓
Célula 5: Handle Missing + Prepare X, y
         ↓
Célula 6: Train/Test Split + StandardScaler
         ↓
Célula 7: Train RandomForest
         ↓
Célula 8: Evaluate (Classification Report, ROC-AUC)
         ↓
Célula 9: Feature Importance (Top 10)
         ↓
Conclusão: Insights + Recomendações
```

### 6. Esperado de Resultados

| Métrica | Range Esperado | Threshold de Atenção |
|---------|---|---|
| **Accuracy** | 0.75 - 0.85 | < 0.70 = overfitting suspeitado |
| **F1-Score** | 0.65 - 0.75 | Melhor que Accuracy devido à minoria |
| **ROC-AUC** | 0.80 - 0.90 | < 0.75 = separabilidade fraca |
| **Precision (Risco)** | 0.60 - 0.75 | Reduzir false positives |
| **Recall (Risco)** | 0.65 - 0.80 | Reduzir false negatives |

**Top Features Esperadas**: DEFASAGEM, FASE_IDEAL, VETERANO, CONCEITOS (PEDRA)

### 7. Garantias de Reprodutibilidade

✅ **Random State=42**: Seed fixa para train_test_split, RandomForest  
✅ **Stratified Split**: Mantém proporção de classes em treino e teste  
✅ **StandardScaler por Fold**: Fit em treino, transform em teste (sem leakage)  
✅ **Copy DataFrame**: Todas as funções retornam cópias  
✅ **Explicit Typing**: Type hints em todas as signatures  

---

## 🔗 Referências

- **Código**: `src/data_cleaning.py`, `src/feature_engineering.py`
- **Visualização**: `scripts/visualization.py`
- **Testes**: `tests/src/test_data_cleaning.py`, `tests/src/test_feature_engineering.py`
- **Dados**: `app/data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx`
- **Output Model**: `app/models/model.pkl` (via `scripts/train.py`)

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from scripts.visualization import plot_feature_importance
from src.data_cleaning import clean_dataframe, load_data
from src.feature_engineering import create_features

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("✅ Imports concluídos")

## 1. Carregamento e Limpeza

In [ ]:
DATA_PATH = "../app/data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

try:
    df = load_data(DATA_PATH)
    print(f"✅ Dataset carregado: {df.shape[0]} linhas × {df.shape[1]} colunas")
except Exception as e:
    print(f"⚠️ Erro ao carregar Excel, usando CSV fallback: {e}")
    df = pd.read_csv("../app/data/processed/df_model_2022.csv")
    print(f"✅ CSV carregado: {df.shape[0]} linhas × {df.shape[1]} colunas")

# Limpeza
df = clean_dataframe(df)
print(f"✅ Após limpeza: {df.shape[0]} linhas")

## 2. Criação do Target

In [ ]:
# Identificar coluna de defasagem
defasagem_col = "Defasagem" if "Defasagem" in df.columns else "DEFASAGEM"

if defasagem_col in df.columns:
    df["target_risco"] = (df[defasagem_col] < 0).astype(int)
    print("✅ Target criado:")
    print(f"   Sem Risco: {(df['target_risco'] == 0).sum()}")
    print(f"   Com Risco: {(df['target_risco'] == 1).sum()}")
else:
    print("⚠️ Coluna de defasagem não encontrada")

## 3. Feature Engineering

In [ ]:
try:
    df = create_features(df)
    print("✅ Features engenheirizadas")
except Exception as e:
    print(f"⚠️ Erro no feature engineering: {e}")
    print("   Continuando com features originais")

## 4. Tratamento de Missing Values

In [ ]:
# Remover colunas com > 70% missing
missing_pct = df.isna().sum() / len(df) * 100
cols_to_drop = missing_pct[missing_pct > 70].index

df = df.drop(columns=cols_to_drop)
print(f"❌ Removidas {len(cols_to_drop)} colunas")

# Preencher com média/moda conforme tipo
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    if col not in ["NOME", "target_risco"]:
        mode_val = df[col].mode()
        df[col] = df[col].fillna(mode_val[0] if len(mode_val) > 0 else "Unknown")

print("✅ Missing values tratados")

## 5. Preparação para Modelagem

In [ ]:
# Separar X e y
X = df.select_dtypes(include=[np.number]).drop(columns=["target_risco"], errors="ignore")
y = df["target_risco"] if "target_risco" in df.columns else None

if y is not None:
    print("✅ Dataset preparado:")
    print(f"   X: {X.shape[0]} × {X.shape[1]}")
    print(f"   y: {len(y)} labels")
    print(f"   Risco: {(y.sum() / len(y) * 100):.1f}%")
else:
    print("⚠️ Target não disponível")

## 6. Train/Test Split

In [ ]:
if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 7. Treinamento

In [ ]:
if y is not None:
    modelo = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    modelo.fit(X_train_scaled, y_train)

    # Predições
    y_pred = modelo.predict(X_test_scaled)
    y_pred_proba = modelo.predict_proba(X_test_scaled)[:, 1]

    print("✅ Modelo treinado")
    print("\n📊 Relatório:")
    print(classification_report(y_test, y_pred, target_names=["Sem Risco", "Risco"]))
    print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

## 8. Feature Importance

In [ ]:
if y is not None:
    feature_importance = pd.DataFrame(
        {"feature": X.columns, "importance": modelo.feature_importances_}
    ).sort_values("importance", ascending=False)

    print("🏆 Top 10 Features:")
    print(feature_importance.head(10).to_string(index=False))

    # Plot
    plot_feature_importance(feature_importance.head(10), title="Top 10 Features")

print("\n✅ Pipeline concluído")